### Import Data

In [ ]:
import polars as pl
from great_tables import GT
import math

events = pl.read_csv(
    "data/2025-10-11.Team.A.@.Team.D.Events.csv",
    schema_overrides={"Player_Id": pl.String},
)

shifts = pl.read_csv("data/2025-10-11.Team.A.@.Team.D.Shifts.csv")

tracking_p1 = pl.read_csv(
    "data/2025-10-11.Team.A.@.Team.D.Tracking_P1.csv",
    schema_overrides={"Rink Location Z (Feet)": pl.Float64},
).drop("Player Id")

tracking_p2 = pl.read_csv(
    "data/2025-10-11.Team.A.@.Team.D.Tracking_P2.csv",
    schema_overrides={"Rink Location Z (Feet)": pl.Float64},
).drop("Player Id")

tracking_p3 = pl.read_csv(
    "data/2025-10-11.Team.A.@.Team.D.Tracking_P3.csv",
    schema_overrides={"Rink Location Z (Feet)": pl.Float64},
).drop("Player Id")


# Convert clock strings ("MM:SS" / "M:SS") to total seconds.
# events.Clock uses zero-padded minutes ("09:30"); tracking "Game Clock" is unpadded ("9:30").
# Casting both to integer seconds makes them directly comparable for joins.
def clock_to_seconds(col):
    parts = pl.col(col).str.split(":")
    return (
        parts.list.get(0).cast(pl.Int64) * 60
        + parts.list.get(1).cast(pl.Int64)
    )


# Add the seconds column and slot it right after its source column.
def add_seconds_after(df, src, new):
    df = df.with_columns(clock_to_seconds(src).alias(new))
    cols = [c for c in df.columns if c != new]
    cols.insert(df.get_column_index(src) + 1, new)
    return df.select(cols)


events = add_seconds_after(events, "Clock", "Clock_Seconds")

tracking_p1 = add_seconds_after(tracking_p1, "Game Clock", "Game_Clock_Seconds")
tracking_p2 = add_seconds_after(tracking_p2, "Game Clock", "Game_Clock_Seconds")
tracking_p3 = add_seconds_after(tracking_p3, "Game Clock", "Game_Clock_Seconds")

### Distribution of Shot Types in the dataset

In [ ]:
shot_types = (
    events.filter(pl.col("Event").is_in(["Shot", "Goal"]))
    .group_by("Detail_1")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / pl.col("count").sum()).alias("pct"))
    .sort("count", descending=True)
)

(
    GT(shot_types)
    .tab_header(title="Distribution of shot types in the dataset")
    .cols_label(Detail_1="Shot Type", pct="% of Shots")
    .fmt_percent("pct", decimals=1)
    .cols_hide("count")
    .cols_width({"Detail_1": "150px", "pct": "110px"})
)


### Distribution of Shot Outcomes in the dataset

In [ ]:
events.filter(pl.col("Event").is_in(["Shot", "Goal"])).select(
    "Event", "Detail_1", "Detail_2", "Detail_3", "Detail_4"
)

In [ ]:
events.filter(pl.col("Event").is_in(["Shot", "Goal"])).get_column(
    "Detail_2"
).unique().sort()

In [ ]:
descriptions = {
    "Saved": "The shot is on target and saved by the goaltender.",
    "Goal": "The shot is on target and not saved by the goaltender, resulting in a goal.",
    "Blocked": "The shot is blocked by a skater from the defending team.",
    "Missed": "The shot is not on target.",
}

shot_outcomes = (
    events.filter(pl.col("Event").is_in(["Shot", "Goal"]))
    .with_columns(
        pl.when(pl.col("Event") == "Goal")
        .then(pl.lit("Goal"))
        .when((pl.col("Event") == "Shot") & (pl.col("Detail_2") == "On Net"))
        .then(pl.lit("Saved"))
        .when((pl.col("Event") == "Shot") & (pl.col("Detail_2") == "Blocked"))
        .then(pl.lit("Blocked"))
        .when((pl.col("Event") == "Shot") & (pl.col("Detail_2") == "Missed"))
        .then(pl.lit("Missed"))
        .alias("Outcome")
    )
    .group_by("Outcome")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / pl.col("count").sum()).alias("pct"))
    .with_columns(pl.col("Outcome").replace(descriptions).alias("Description"))
    .select("Outcome", "Description", "pct")
    .sort("pct", descending=True)
)

(
    GT(shot_outcomes)
    .tab_header(title="Distribution of shot outcomes in the dataset")
    .cols_label(
        Outcome="Shot Outcome",
        Description="Description",
        pct="% of Shots",
    )
    .fmt_percent("pct", decimals=1)
    .cols_width({"Outcome": "130px", "Description": "420px", "pct": "110px"})
)


### Feature-Based Expected Goals Model

#### Visualize Shot Location

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Arc, Circle, FancyBboxPatch

# Prepare shot data with outcome labels, normalized coordinates, and shot angle
shot_data = (
    events
    .filter(
        pl.col("Event").is_in(["Shot", "Goal"]),
        pl.col("Period") == 1,
    )
    .with_columns(
        pl.when(pl.col("Event") == "Goal")
        .then(pl.lit("Goal"))
        .when((pl.col("Event") == "Shot") & (pl.col("Detail_2") == "On Net"))
        .then(pl.lit("Saved"))
        .when((pl.col("Event") == "Shot") & (pl.col("Detail_2") == "Blocked"))
        .then(pl.lit("Blocked"))
        .when((pl.col("Event") == "Shot") & (pl.col("Detail_2") == "Missed"))
        .then(pl.lit("Missed"))
        .alias("Outcome")
    )
    # Normalize all shots to the right side
    .with_columns(
        pl.when(pl.col("X_Coordinate") < 0)
        .then(-pl.col("X_Coordinate"))
        .otherwise(pl.col("X_Coordinate"))
        .alias("X_Coordinate"),

        pl.when(pl.col("X_Coordinate") < 0)
        .then(-pl.col("Y_Coordinate"))
        .otherwise(pl.col("Y_Coordinate"))
        .alias("Y_Coordinate"),
    )
    # Compute shot angle: 0° = straight on, 90° = side of net
    .with_columns(
        (
            pl.arctan2(pl.col("Y_Coordinate"), 89 - pl.col("X_Coordinate"))
            .abs()
            * (180 / 3.141592653589793)
        ).alias("Shot_Angle")
    )
    .select("X_Coordinate", "Y_Coordinate", "Outcome", "Shot_Angle")
)

outcome_markers = {
    "Goal":    {"marker": "*", "size": 250},
    "Saved":   {"marker": "o", "size":  90},
    "Blocked": {"marker": "s", "size":  90},
    "Missed":  {"marker": "^", "size":  90},
}

cmap = plt.cm.plasma
angles = shot_data.get_column("Shot_Angle").to_list()
norm = mcolors.Normalize(vmin=min(angles), vmax=max(angles))

fig, ax = plt.subplots(figsize=(10, 8))
ax.set_facecolor("#e8f4fb")

# --- Rink ---
ax.add_patch(FancyBboxPatch(
    (-100, -42.5), 200, 85,
    boxstyle="round,pad=0,rounding_size=28",
    linewidth=2, edgecolor="black", facecolor="white", zorder=1,
))
for x, color, lw in [(0, "#c0392b", 3.0), (25, "#2980b9", 3.0), (89, "#c0392b", 1.5)]:
    ax.plot([x, x], [-42.5, 42.5], color=color, linewidth=lw, zorder=2)
for y in [22, -22]:
    ax.add_patch(Circle((69, y), 15, fill=False, color="#c0392b", linewidth=1.5, zorder=2))
ax.add_patch(Arc((89, 0), 12, 12, theta1=270, theta2=90, color="#2980b9", linewidth=1.5, zorder=2))

# --- Plot shots: color = angle, shape = outcome ---
for outcome, style in outcome_markers.items():
    subset = shot_data.filter(pl.col("Outcome") == outcome)
    ax.scatter(
        subset.get_column("X_Coordinate").to_list(),
        subset.get_column("Y_Coordinate").to_list(),
        c=subset.get_column("Shot_Angle").to_list(),
        cmap=cmap, norm=norm,
        marker=style["marker"], s=style["size"],
        zorder=5, edgecolors="black", linewidths=0.5, alpha=0.9,
    )

# --- Colorbar ---
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label("Shot Angle (°)", fontsize=11)

# --- Legend for shapes ---
handles = [
    plt.scatter([], [], marker=s["marker"], s=s["size"], c="gray",
                edgecolors="black", linewidths=0.5, label=o)
    for o, s in outcome_markers.items()
]
ax.legend(handles=handles, loc="upper left", fontsize=10, frameon=False, title="Outcome")

ax.set_xlim(0, 105)
ax.set_ylim(-47, 55)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Shot Locations — Period 1\nColor = Shot Angle, Shape = Outcome", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()


#### 2.4.1.1 Shot Location

In [ ]:
shot_data.with_columns(
    (
        pl.arctan2(pl.col("Y_Coordinate"), pl.col("X_Coordinate") - 89)
        .abs()
        * (180 / 3.141592653589793)
    ).alias("Shot_Angle")
)

In [ ]:

(
    events
    .filter(pl.col("Event").is_in(["Shot", "Goal"]))
    # Normalize to right side
    .with_columns(
        pl.when(pl.col("X_Coordinate") < 0)
        .then(-pl.col("X_Coordinate"))
        .otherwise(pl.col("X_Coordinate"))
        .alias("X_Coordinate"),

        pl.when(pl.col("X_Coordinate") < 0)
        .then(-pl.col("Y_Coordinate"))
        .otherwise(pl.col("Y_Coordinate"))
        .alias("Y_Coordinate"),
    )
    # Shot angle (degrees) and meridian distance
    .with_columns(
        (
            pl.arctan2(pl.col("Y_Coordinate"), 89 - pl.col("X_Coordinate"))
            .abs()
            * (180 / math.pi)
        ).alias("Shot_Angle"),
        pl.col("Y_Coordinate").abs().alias("d_m"),
    )
    # Distance via trigonometry: d = d_m / sin(θ), fallback to (89 - x) when Y = 0
    .with_columns(
        pl.when(pl.col("d_m") == 0)
        .then(89 - pl.col("X_Coordinate"))
        .otherwise(
            pl.col("d_m") / (pl.col("Shot_Angle") * math.pi / 180).sin()
        )
        .alias("d_trig"),
        # Euclidean distance as sanity check
        ((89 - pl.col("X_Coordinate")).pow(2) + pl.col("Y_Coordinate").pow(2))
        .sqrt()
        .alias("d_euclidean"),
    )
    .select("X_Coordinate", "Y_Coordinate", "Shot_Angle", "d_m", "d_trig", "d_euclidean")
)

#### Shooter Motion

In [ ]:
# Goal: the shooter's *speed* (magnitude of velocity) at the instant they shot.
# Tracking gives position over time; events tell us who/when/where each shot was.
#
# Tracking runs at ~30 fps: the "Image Id" suffix is a per-period frame counter that
# ticks ~30 times per game-clock second (e.g. "..._066486" -> "..._066487"). Frames
# occasionally drop, so velocity uses the *actual* frame-index gap for dt, never an
# assumed constant. We smooth with a small central difference to damp tracking jitter.

FPS = 30                  # tracking frame rate (frames per second)
SMOOTH_K = 3              # central-difference half-window (~0.1 s each side)
FT_PER_S_TO_MPH = 0.6818  # 1 ft/s = 0.6818 mph
MAX_SPEED_MPH = 30.0      # human skaters peak ~25 mph; above this is a tracking glitch
MAX_SPEED_FPS = MAX_SPEED_MPH / FT_PER_S_TO_MPH

# One table of every tracked player-frame across the three periods.
tracking = pl.concat([tracking_p1, tracking_p2, tracking_p3])

players = (
    tracking
    .filter(pl.col("Player or Puck") == "Player")
    .with_columns(
        # Numeric frame index from the Image Id suffix, and jersey as a string for matching.
        pl.col("Image Id").str.split("_").list.last().cast(pl.Int64).alias("Frame"),
        pl.col("Player Jersey Number").cast(pl.String).alias("Jersey"),
    )
    # Drop rows whose jersey couldn't be read: they'd all collapse into one bogus "null"
    # track, and differencing positions of *different* players invents huge fake speeds.
    .filter(pl.col("Jersey").is_not_null())
    # Order each player's frames in time so shift() walks consecutive samples.
    .sort(["Period", "Team", "Jersey", "Frame"])
    .with_columns(
        # Central difference over +/- SMOOTH_K frames, partitioned per player-period.
        (pl.col("Rink Location X (Feet)").shift(-SMOOTH_K) - pl.col("Rink Location X (Feet)").shift(SMOOTH_K))
            .over(["Period", "Team", "Jersey"]).alias("dx"),
        (pl.col("Rink Location Y (Feet)").shift(-SMOOTH_K) - pl.col("Rink Location Y (Feet)").shift(SMOOTH_K))
            .over(["Period", "Team", "Jersey"]).alias("dy"),
        (pl.col("Frame").shift(-SMOOTH_K) - pl.col("Frame").shift(SMOOTH_K))
            .over(["Period", "Team", "Jersey"]).alias("dframe"),
    )
    .with_columns(
        # dt = frames / fps (seconds); speed = displacement / dt, in feet per second.
        ((pl.col("dx") ** 2 + pl.col("dy") ** 2).sqrt() / (pl.col("dframe") / FPS)).alias("speed_fps")
    )
    .with_columns(
        # Null out physically impossible speeds (tracking ID-swaps / position jumps) so
        # they don't poison the per-shot pick or its fallback median.
        pl.when(pl.col("speed_fps") <= MAX_SPEED_FPS).then(pl.col("speed_fps")).otherwise(None).alias("speed_fps")
    )
    .with_columns(
        (pl.col("speed_fps") * FT_PER_S_TO_MPH).alias("speed_mph")
    )
)

players.select(
    "Period", "Team", "Jersey", "Frame", "Game_Clock_Seconds",
    "Rink Location X (Feet)", "Rink Location Y (Feet)", "speed_fps", "speed_mph",
)


In [ ]:
# Match each Shot/Goal event to the tracking frame where the puck was released, then
# read the shooter's speed there.
#
# Matching keys: events Player_Id is a jersey number, but jerseys repeat across teams,
# so we match on Period + Team + Jersey. Event "Team" is a name (e.g. "Team A");
# tracking "Team" is Home/Away -> map via Home_Team/Away_Team.
#
# Pinning the release: the event stores the release X/Y. Within +/-1 game-clock second
# of the event we pick the shooter frame whose tracked position is closest to that
# release point. Event and tracking share the same rink frame here, but to be safe we
# also test the sign-flipped coords (-X,-Y) and keep whichever is closer.

shots = (
    events.filter(pl.col("Event").is_in(["Shot", "Goal"]))
    .with_columns(
        pl.when(pl.col("Team") == pl.col("Home_Team")).then(pl.lit("Home"))
          .otherwise(pl.lit("Away")).alias("Track_Team")
    )
    .with_row_index("shot_id")
)

# Candidate shooter frames: same player, within the event second (+/-1s), valid position.
cand = (
    shots.join(
        players.select([
            "Period", "Team", "Jersey", "Game_Clock_Seconds", "Frame",
            "Rink Location X (Feet)", "Rink Location Y (Feet)", "speed_fps",
        ]),
        left_on=["Period", "Track_Team", "Player_Id"],
        right_on=["Period", "Team", "Jersey"], how="left",
    )
    .filter((pl.col("Game_Clock_Seconds") - pl.col("Clock_Seconds")).abs() <= 1)
    .filter(pl.col("Rink Location X (Feet)").is_not_null()
            & pl.col("Rink Location Y (Feet)").is_not_null())
    .with_columns(
        ((pl.col("Rink Location X (Feet)") - pl.col("X_Coordinate")) ** 2
         + (pl.col("Rink Location Y (Feet)") - pl.col("Y_Coordinate")) ** 2).sqrt().alias("d_direct"),
        ((pl.col("Rink Location X (Feet)") + pl.col("X_Coordinate")) ** 2
         + (pl.col("Rink Location Y (Feet)") + pl.col("Y_Coordinate")) ** 2).sqrt().alias("d_flip"),
    )
    .with_columns(
        pl.min_horizontal("d_direct", "d_flip").alias("match_dist"),
        (pl.col("d_flip") < pl.col("d_direct")).alias("used_flip"),
    )
)

# Release frame = the candidate closest to the release point (nulls_last so a real
# match is never beaten by a missing distance).
release = (
    cand.sort("match_dist", nulls_last=True)
    .group_by("shot_id", maintain_order=True)
    .first()
    .select(["shot_id", "Frame", "match_dist", "used_flip", "speed_fps"])
)

# Fallback: representative speed over the whole event-second window.
window_med = cand.group_by("shot_id").agg(
    pl.col("speed_fps").median().alias("speed_fps_window_median"),
    pl.len().alias("n_candidates"),
)

# A match is trusted when the closest frame is within ~10 ft and has a defined speed;
# otherwise fall back to the window median (flagged), or mark no tracking at all.
MAX_MATCH_DIST = 10.0
shot_speeds = (
    shots.join(release, on="shot_id", how="left")
    .join(window_med, on="shot_id", how="left")
    .with_columns(
        pl.when((pl.col("match_dist") <= MAX_MATCH_DIST) & pl.col("speed_fps").is_not_null())
          .then(pl.col("speed_fps"))
          .otherwise(pl.col("speed_fps_window_median"))
          .alias("shooter_speed_fps"),
        pl.when((pl.col("match_dist") <= MAX_MATCH_DIST) & pl.col("speed_fps").is_not_null())
          .then(pl.lit("matched"))
          .when(pl.col("speed_fps_window_median").is_not_null())
          .then(pl.lit("fallback_window_median"))
          .otherwise(pl.lit("no_tracking"))
          .alias("match_quality"),
    )
    .with_columns((pl.col("shooter_speed_fps") * 0.6818).alias("shooter_speed_mph"))
    .sort(["Period", "Clock_Seconds"], descending=[False, True])
)

print(shot_speeds["match_quality"].value_counts())
shot_speeds.select(
    "Period", "Clock", "Team", "Player_Id", "Event", "Detail_1",
    "Frame", "match_dist", "used_flip", "shooter_speed_fps", "shooter_speed_mph", "match_quality",
).head()


In [ ]:
# Shooter speed at the moment of each shot, in ft/s and mph.
shot_speed_table = shot_speeds.select(
    "Period", "Clock", "Team", "Player_Id", "Event", "Detail_1",
    "shooter_speed_fps", "shooter_speed_mph", "match_quality",
)

(
    GT(shot_speed_table)
    .tab_header(
        title="Shooter speed at the moment of the shot",
        subtitle="Magnitude of the shooter's velocity from tracking, pinned to the release frame",
    )
    .cols_label(
        Player_Id="Jersey",
        Detail_1="Shot Type",
        shooter_speed_fps="Speed (ft/s)",
        shooter_speed_mph="Speed (mph)",
        match_quality="Match",
    )
    .fmt_number(["shooter_speed_fps", "shooter_speed_mph"], decimals=1)
    .cols_width({
        "Period": "70px", "Clock": "70px", "Team": "90px", "Player_Id": "70px",
        "Event": "70px", "Detail_1": "110px",
        "shooter_speed_fps": "110px", "shooter_speed_mph": "110px", "match_quality": "150px",
    })
)


In [ ]:
# Sanity check: trace one shooter through the event second and mark the release frame
# we picked, so we can eyeball that the pin lands at the event's release location.
ex = shot_speeds.filter(
    (pl.col("Period") == 1) & (pl.col("Clock") == "19:26") & (pl.col("Player_Id") == "29")
).row(0, named=True)

track = (
    players.filter(
        (pl.col("Period") == ex["Period"]) & (pl.col("Team") == ex["Track_Team"])
        & (pl.col("Jersey") == ex["Player_Id"])
        & ((pl.col("Game_Clock_Seconds") - ex["Clock_Seconds"]).abs() <= 1)
    ).sort("Frame")
)

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(
    track["Rink Location X (Feet)"], track["Rink Location Y (Feet)"],
    c=track["speed_fps"], cmap="viridis", s=40, zorder=3,
)
# Event release location (orange X) and the tracking frame we pinned it to (red ring).
ax.scatter([ex["X_Coordinate"]], [ex["Y_Coordinate"]], marker="x", s=200,
           color="orange", linewidths=3, zorder=5, label="event release X/Y")
rel = track.filter(pl.col("Frame") == ex["Frame"])
ax.scatter(rel["Rink Location X (Feet)"], rel["Rink Location Y (Feet)"],
           s=260, facecolors="none", edgecolors="red", linewidths=2.5,
           zorder=4, label="picked release frame")
plt.colorbar(sc, ax=ax, label="speed (ft/s)")
ax.legend(loc="best", fontsize=9)
ax.set_aspect("equal")
ax.set_xlabel("Rink X (ft)"); ax.set_ylabel("Rink Y (ft)")
ax.set_title(
    f"Shooter #{ex['Player_Id']} ({ex['Team']}) — P{ex['Period']} {ex['Clock']}\n"
    f"speed at release = {ex['shooter_speed_fps']:.1f} ft/s "
    f"({ex['shooter_speed_mph']:.1f} mph), match_dist = {ex['match_dist']:.1f} ft",
    fontsize=11,
)
plt.tight_layout()
plt.show()
